# Detection

In [ ]:
import torch
import numpy as np
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from pathlib import Path
import torch
from torch import nn, optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader, Dataset
from datetime import datetime
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import torch.nn.functional as F
from collections import Counter
import json
from itertools import product
import math
import hashlib
import io
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

### Setup

In [ ]:
H_in, W_in = 48, 60
SEED = 265
DATA_DIR = Path("data")
BATCH_SIZE = 32

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Training on device {device}.")

In [ ]:
DO_TRAINING = True  # Set me to TRUE to run training from the start, otherwise we just load existing models

In [ ]:
save_dir = Path("imgs/object_detection")
save_dir.mkdir(parents=True, exist_ok=True)

### Load data and preprocessing

### Normalize Images

In [ ]:
class Preprocessor:
    def __init__(self):
        self.transformer = None
        self.mean = None
        self.std = None

    def fit(self, dataset):
        train_loader = DataLoader(dataset, batch_size=BATCH_SIZE)

        mean = 0.0
        std = 0.0
        total_images = 0

        for images, _ in train_loader:
            batch_samples = images.size(0)
            images = images.view(batch_samples, images.size(1), -1)
            mean += images.mean(2).sum(0)
            std += images.std(2).sum(0)
            total_images += batch_samples

        self.mean = mean / total_images
        self.std = std / total_images

        self.transformer = transforms.Compose(
           transforms.Normalize(self.mean, self.std)
        )
        return self.transformer

    def process(self, data):
        return self.transformer(data)

    def unnormalize(self, img):
        img = img.clone()
        for c in range(img.shape[0]):
            img[c] = img[c] * self.std[c] + self.mean[c]
        return torch.clamp(img, 0, 1)

In [ ]:
class MNISTAugmented(Dataset):
    """
    Since we are given train, val, test we can make one instance of the classes each dealing with the different subset of the data
    """
    def __init__(self, data_path: Path, split: str, transform=None):
        dataset = torch.load(data_path / f"localization_{split}.pt")

        self.data = dataset.data
        self.targets = dataset.targets
        self.transform = transform

    def set_transform(self, transform):
        self.transform = transform

    def __getitem__(self, idx):
        img = self.data[idx]
        target = self.targets[idx]

        if self.transform:
            img = self.transform(img)

        return img, target

    def __len__(self):
        return len(self.data)

In [ ]:
def load_mnist(data_dir, preprocessor):
    train_dataset = MNISTAugmented(data_dir, split="train")
    val_dataset = MNISTAugmented(data_dir, split="val")
    test_dataset = MNISTAugmented(data_dir, split="test")

    preprocessor.fit(train_dataset)

    train_dataset.set_transform(preprocessor.transformer)
    val_dataset.set_transform(preprocessor.transformer)
    test_dataset.set_transform(preprocessor.transformer)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    return train_loader, val_loader, test_loader

mnist = MNISTAugmented(DATA_DIR)
processor = Preprocessor()

train_loader, val_loader, test_loader = load_mnist(mnist = mnist, preprocessor=processor)


In [ ]:
def show_img(img, label, title=None) -> None:
    img = img.squeeze().permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    plt.title(f"Label: {label}")
    plt.axis("off")
    if title:
        plt.title(title)
        plt.savefig(save_dir / f"{title}.png")
    plt.show()

In [ ]:
batch = next(iter(train_loader))
imgs, labels = batch
show_img(imgs[0], labels[0], title="example_plane_train")  # plane = label 0
show_img(imgs[3], labels[3], title="example_bird_train")  # bird = label 1

# Utils

In [ ]:
def train(
    n_epochs,
    optimizer,
    model,
    loss_fn,
    train_loader: DataLoader,
    validation_loader: DataLoader,
):
    n_batch_train = len(train_loader)
    n_batch_val = len(validation_loader)

    losses_train, losses_val = [], []
    train_accuracies, val_accuracies = {}, {}

    model.train()  # set to train mode
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(1, n_epochs + 1):
        loss_train, loss_val = 0.0, 0.0
        for imgs, labels in train_loader:
            imgs = imgs.to(device=device, dtype=torch.double)  # dtype=torch.double
            labels = labels.to(device=device)

            outputs = model(imgs)

            loss = loss_fn(outputs, labels)
            loss.backward()

            optimizer.step()
            optimizer.zero_grad()

            loss_train += loss.item()

        model.eval()  # observe model performance
        with torch.no_grad():
            for val_imgs, val_labels in validation_loader:
                imgs_val = val_imgs.to(device=device)  # dtype=torch.double)
                val_labels = val_labels.to(device=device)  # dtype=torch.double)
                val_outputs = model(imgs_val)
                val_loss = loss_fn(val_outputs, val_labels)
                loss_val += val_loss.item()

        # if epoch in EPOCH_INVESTIGATE_POINTS:
        #     train_accuracies[epoch] = compute_accuracy(model, train_loader)
        #     val_accuracies[epoch] = compute_accuracy(model, validation_loader)

        # swap back to train mode
        model.train()

        losses_train.append(loss_train / n_batch_train)
        losses_val.append(loss_val / n_batch_val)

        if epoch == 1 or epoch % 10 == 0:
            print(
                "{}  |  Epoch {}  |  Training loss {:.3f}".format(
                    datetime.now().time(), epoch, loss_train / n_batch_train
                )
            )
    return losses_train, losses_val, train_accuracies, val_accuracies

### Training

In [ ]:
class CNNBaseline(nn.Module):
    """
    inp layer: conv2d
    hid layer: pool
    hid layer: conv2
    hid layer: conv2
    hid layer: fc1
    hid layer: fc2
    hid layer: fc3

    out layer: 32 - 2 no activation function
    """
    # todo add padding
    
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(48 * 60, 6, 2)
        self.pool = nn.MaxPool2d(2, 2) # turn 48*60 to 24*30
        # self.conv2 = nn.Conv2d(6, 16, 5)
        # self.fc1 = nn.Linear(16 * 5 * 5, 120)
        # self.fc2 = nn.Linear(120, 84)
        # self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
class CNNDeep(nn.Module):
    """
    inp layer: conv2d
    hid layer: pool
    hid layer: conv2
    hid layer: conv2
    hid layer: fc1
    hid layer: fc2
    hid layer: fc3

    out layer: 32 - 2 no activation function
    """
        
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(.parameters(), lr=0.001, momentum=0.9)

### Prediction

In [ ]:
# TODO

### Model selection and evaluation

In [ ]:
# TODO

In [ ]:
def get_map_results(model, eval_loader, device):
    '''
        Helper functions to get predictions and targets in the format required for mAP calculation.
        Depending on your data processing and model architecture this function can either be used as is, 
        modified to fit your needs or used as a blue print for a rewrite.
        Here it is assussmed that the image has been divide into a 2 x 3 grid.
        ----------------------------------------------------------
        Run through the data in the dataloader and collect predicitions and targets for mAP calculation.

        torchmetric mAP expects predictions and targets in the format:
        preds = [
           { "boxes": tensor([[x1, y1, x2, y2], ...]), "scores": tensor([score1, score2, ...]), "labels": tensor([label1, label2, ...])},
            ...   ]
        and targets = [
            { "boxes": tensor([[x1, y1, x2, y2], ...]), "labels": tensor([label1, label2, ...])},
            ...   ]
        where each dict in the list corresponds to one image in the dataset and contains the predicted and true results
    '''
    model.eval()
    with torch.no_grad():
        preds = []
        targets = []
        for images, labels in eval_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            outputs = outputs.permute(0, 2, 3, 1)                               # (B, 7, 2, 3) → (B, 2, 3, 7)
            outputs = outputs.reshape(outputs.shape[0], -1, outputs.shape[-1])  # (B, 2, 3, 7) → (B, 6, 7)
            labels = labels.reshape(labels.shape[0], -1, labels.shape[-1])      # (B, 2, 3, 6) → (B, 6, 6)
            for output, label in zip(outputs, labels):
                pred_boxes = []
                pred_scores = []
                pred_labels = []
                target_boxes = []
                target_labels = []
                # collect predicted boxes, scores and labels for the current image
                for i, cell_output in enumerate(output):
                    pred_object_presence = (torch.sigmoid(cell_output[0]) > 0.5) * 1.0
                    if pred_object_presence == 1:
                        # get propability of object presence and class probabilities to compute detection score for mAP calculation
                        obj_prop = torch.sigmoid(cell_output[0]).item()
                        class_prop = F.softmax(cell_output[5:], dim=0)
                        pred_label = torch.argmax(class_prop)
                        detect_score = obj_prop * class_prop[pred_label]
                        # convert from local to global coordinates before we can compare with the labels and compute IoU for mAP calculation
                        bbox_global = local_to_global(i // 3, i % 3, cell_output[1:5])
                        bbox_xyxy = xywh_to_xyxy(bbox_global)
                        bbox_xyxy = torch.stack(bbox_xyxy)
                        # collect predicted boxes, scores and labels for the current image
                        pred_boxes.append(bbox_xyxy)
                        pred_scores.append(detect_score)
                        pred_labels.append(pred_label)
                # collect true boxes and labels for the current image
                for i, cell_label in enumerate(label):
                    true_object_presence = cell_label[0]
                    if true_object_presence == 1:
                        bbox_global = local_to_global(i // 3, i % 3, cell_label[1:5])
                        bbox_xyxy = xywh_to_xyxy(bbox_global)
                        bbox_xyxy = torch.stack(bbox_xyxy)
                        target_boxes.append(bbox_xyxy)
                        target_labels.append(int(cell_label[-1]))
                # store predictions and targets for the current image in the format required for mAP calculation
                # if there are no predicted boxes, we need to create an empty tensor for the boxes, scores and labels to avoid errors in the mAP calculation
                if len(pred_boxes) == 0:
                    pred_dict = {
                        "boxes": torch.zeros((0, 4), device=device),
                        "scores": torch.zeros((0,), device=device),
                        "labels": torch.zeros((0,), dtype=torch.long, device=device),
                    }
                    preds.append(pred_dict)
                else:
                    pred_dict = {
                        "boxes": torch.stack(pred_boxes),
                        "scores": torch.tensor(pred_scores, device=device),
                        "labels": torch.tensor(pred_labels, device=device),
                    }
                    preds.append(pred_dict)
                # if there are no true boxes, we need to create an empty tensor for the boxes and labels to avoid errors in the mAP calculation            
                if len(target_boxes) == 0:
                    target_dict = {
                        "boxes": torch.zeros((0, 4), device=device),
                        "labels": torch.zeros((0,), dtype=torch.long, device=device),
                    }
                    targets.append(target_dict)
                else:
                    target_dict = {
                        "boxes": torch.stack(target_boxes),
                        "labels": torch.tensor(target_labels, device=device),
                    }
                    targets.append(target_dict)
    
    # compute mAP using torchmetrics
    metric = MeanAveragePrecision(iou_type="bbox")
    metric.update(preds, targets)
    results = metric.compute()
    # results is a dict with the mAP results for different IoU thresholds and the overall mAP
    return results        

def local_to_global(i, j, bb, width=60, height=48, cols=3, rows=2):
    x, y, w, h = bb
    # get the dimensions of a single grid cell
    cell_width, cell_height = width / cols, height / rows
    # convert from local to global coordinates
    global_x = x * cell_width + j * cell_width
    global_y = y * cell_height + i * cell_height
    global_w = w * cell_width
    global_h = h * cell_height

    return global_x, global_y, global_w, global_h

def xywh_to_xyxy(bb):
    # convert from center format to box format
    x_center, y_center, w, h = bb
    x1 = x_center - w/2
    y1 = y_center - h/2
    x2 = x_center + w/2
    y2 = y_center + h/2
    return x1, y1, x2, y2   